# Problem Set: Reviewing rail-ontology change requests with a reasoner and Claude
### Chapter 3 — Description Logics

**Topic coverage:** Keet §3.1 (ALC, the tableau), §3.2 (the DL naming scheme and what each letter costs), §3.3 (reasoning services by reduction to satisfiability); course notebooks 01–04 of this chapter. Tools: DSPy + GEPA, skills and the self-improvement loop, LangChain agents, value iteration on a stochastic MDP, and the Chapter 3 tableau reasoner.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 20 / Part B 29 / Part C 31 / Part D 20); 54 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$6–15 estimated on `claude-opus-5` for a full run (≈ 270 DSPy calls at roughly \$0.02–0.05 each, plus 8 agent episodes at ≈ \$0.10–0.25 each; GEPA and the self-improvement rounds are the largest lines). Nothing in this estimate has been measured — your cost lines are. Re-running unchanged cells is free: DSPy caches identical requests. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

**Meridian Rail**, a (fictional) railway infrastructure manager, keeps its asset
information in a modular OWL ontology. Discipline teams — track, signalling,
structures, power, stations — submit new modules as pull requests to the
**Ontology Change Board**, which applies two gates before a module is merged:

* **Expressivity.** The nightly classification job runs a lightweight reasoner.
  A module that needs transitive roles, a role hierarchy, inverse roles or number
  restrictions must be routed to the heavyweight reasoner job instead, so every
  request must state the **description logic the module needs** — ALC, S, ALCHIQ, …
* **The reviewer's question.** Each request carries one question from the
  discipline lead — *"is every level-crossing barrier a safety-critical asset?"* —
  which is a **subsumption** query against the module. The board wants a verified
  answer, not an opinion.

The board has asked you, the ontology engineer, to automate the review:

1. **Understand the tools you already have.** A cheap *told-subsumption* heuristic
   and the Chapter 3 tableau. The tableau is a **free, always-right oracle** on
   this corpus — which makes this the one chapter where a deployed system can
   label its own failures with no human in the loop. Find out exactly where
   "always right" stops.
2. **Build the grader and a Claude reviewer**, measure it on a development split.
3. **Optimise it honestly** with GEPA (train / dev / held-out test, cost, noise),
   then let it **improve itself on unlabelled production traffic** that the
   oracle labels.
4. **Ship a review agent** that calls the reasoner instead of guessing, and
   **price the reasoner**: CI minutes are budgeted, so decide — per request — when
   the expensive, sound check is worth it over the cheap heuristic.

The corpus (24 labelled modules, split 8/8/8 by item — one per phenomenon per
split, four "subsumed" and four "not subsumed" in every split — plus 6 unlabelled
field requests), the review guidelines, the agent's tools and the budget MDP are
provided in `ch03_agentic.py`.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `05_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 3 — Description Logics — Reviewing rail-ontology change requests with a reasoner and Claude')

import re, statistics as st
import dspy
import numpy as np
import pandas as pd
import ch03_toolkit as dl
import ch03_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents
from oe_course.skills import Skill
from oe_course.selfimprove import SelfImprovingSkill

A = dl.Atomic
train, dev, test = AG.build_dataset("train"), AG.build_dataset("dev"), AG.build_dataset("test")
CERTIFIED = AG.validate_gold()          # every label re-derived by the engine (< 1 s)
print(f"train {len(train)} / dev {len(dev)} / test {len(test)} / field {len(AG.FIELD_CASES)}")

### The corpus

Every row is one change request. `kb` is what the reviewer (human or Claude) sees — the
module's axioms and role facts, rendered *without* naming the logic — and `query` is the
discipline lead's question. The gold labels were written by hand and then re-derived by
`AG.validate_gold()`, which also tells you **how** each verdict is certified.

In [ ]:
overview = pd.DataFrame([{"id": ex.id, "split": ex.split, "phenomenon": ex.phenomenon,
                          "gold_dl": ex.gold_dl, "subsumed": ex.gold_subsumption,
                          "certified by": CERTIFIED[ex.id]} for ex in train + dev + test])
print(overview.groupby("split").subsumed.agg(["count", "sum"]).rename(columns={"sum": "n_subsumed"}))
overview

In [ ]:
ex = dev[0]
print(ex.kb, "\n")
print(ex.query)

---

## Part A — The cheap heuristic, the free oracle, and where it stops being right

Before anything touches a model, understand the two answers you can already compute for
free: a structural heuristic, and the tableau.

### A1. A told-subsumption heuristic, and a dataset that can punish it (10 pts)

Reviewers' first instinct is to *read the hierarchy*: follow the asserted superclasses
upwards and see whether the queried class turns up. Implement that as
`told_subsumes(tbox, sub, sup) -> bool`:

* `sub` is told-subsumed by itself;
* for an axiom whose **left side is atomic** `X`, every **atomic conjunct** of the right
  side (flatten nested `dl.And`) is a told superclass of `X`; anything else on the right
  (`exists`, `forall`, `or`, `not`, number restrictions) is ignored;
* an **equivalence** `X == Y` with *both* sides atomic works in both directions;
* take the transitive closure.

Then build `a1`, one row per labelled item (train + dev + test), with columns `id`,
`split`, `phenomenon`, `gold`, `told` (your heuristic's verdict), `heuristic_correct` and
`risk_class` (`AG.risk_class(tbox)`), and `a1_acc`: indexed by split, with columns
`heuristic` (your heuristic's accuracy) and `always_subsumed` (the accuracy of the trivial
strategy *"always answer subsumed"*).

In writing: (i) the heuristic can only err in one direction — which, and why? (ii) The
old version of this corpus had almost only "subsumed" questions. What would *"always
answer subsumed"* have scored there, and what would that have meant for an optimiser whose
feedback includes the guideline `use-the-reasoner`?

In [ ]:
def told_subsumes(tbox: dl.TBox, sub: str, sup: str) -> bool:
    # TODO
    raise NotImplementedError

a1 = None        # TODO: one row per labelled item
a1_acc = None    # TODO: per split: heuristic accuracy, always-subsumed accuracy

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=5):
    T = dl.TBox()
    T.add(A("X"), dl.And(A("Y"), dl.And(A("Z"), dl.Exists("r", A("W")))))
    T.add(A("Z"), A("P"))
    T.add(A("P"), A("Q"), equivalence=True)
    T.add(A("K"), dl.And(A("X"), A("Y")), equivalence=True)
    assert told_subsumes(T, "X", "X") and told_subsumes(T, "X", "Z") and told_subsumes(T, "X", "Q")
    assert told_subsumes(T, "Q", "P"), "an atomic equivalence works both ways"
    assert not told_subsumes(T, "X", "W") and not told_subsumes(T, "Y", "X")
    assert not told_subsumes(T, "X", "K"), "a complex definition is not a told superclass"
    for c in AG.CASES + AG.FIELD_CASES:          # a told subsumption is always a real one
        t = c.tbox()
        if told_subsumes(t, c.sub, c.sup):
            assert dl.subsumes(A(c.sub), A(c.sup), t), f"{c.id}: heuristic is unsound"
    assert isinstance(a1, pd.DataFrame) and len(a1) == 24
    assert {"id", "split", "phenomenon", "gold", "told", "heuristic_correct", "risk_class"} <= set(a1.columns)
    assert (a1.heuristic_correct == (a1.told == a1.gold)).all()
    assert set(a1_acc.index) == {"train", "dev", "test"}
    assert np.allclose(a1_acc["always_subsumed"], 0.5)
    assert np.allclose(a1_acc["heuristic"], a1.groupby("split").heuristic_correct.mean()[a1_acc.index])

**Your answer:**

*(Write your answer here.)*

### A2. Where the free oracle stops being right (10 pts)

The tableau decides **ALC**. It *names* S, H, I, N and Q but reasons as if transitivity,
sub-roles, inverses and number restrictions were not there — so it reasons over a
*weaker* theory than the module states.

(a) Build `blind_spot = (tbox, sub, sup)`: a small rail module with a **role hierarchy or
a transitive role** in which `sub ⊑ sup` **is** entailed but `dl.subsumes` says it is
not. Then build `compiled`: a copy of the module plus one or more ALC axioms that the role
box licenses (see `AG.rbox_licensed` for the three accepted schemata), after which the
tableau finds the subsumption.

(b) `AG.validate_gold()` certifies "not subsumed" on the non-ALC modules with
countermodels (`AG.CERTIFICATES`). Write your own certificate `my_certificate` — an
`AG.Interpretation` — for a **pure-ALC** "not subsumed" item of your choice
(`cert_case`, one not in `AG.CERTIFICATES`), and confirm it with `AG.is_countermodel`.

(c) In writing: in which direction is the tableau's verdict trustworthy on *any* module
of this corpus, and why? What does (a) mean for a system that labels its own training
data with this oracle? What would you run in production instead?

In [ ]:
blind_spot = None       # TODO (a): (tbox, sub, sup)
compiled = None         # TODO (a): the module plus RBox-licensed ALC axioms

cert_case = "..."       # TODO (b)
my_certificate = None   # TODO (b): AG.Interpretation(domain=..., concepts=..., roles=...)

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=4):
    t, sub, sup = blind_spot
    assert t.role_hierarchy or t.transitive_roles, "the blind spot needs an RBox feature"
    assert not dl.subsumes(A(sub), A(sup), t), "the tableau must miss it"
    assert dl.subsumes(A(sub), A(sup), compiled), "the compiled module must expose it"
    extra = [ax for ax in compiled.axioms if ax not in t.axioms]
    assert extra and all(ax in compiled.axioms for ax in t.axioms)
    assert all(AG.rbox_licensed(ax, t) for ax in extra), "only RBox-licensed axioms may be added"
    c = AG.case(cert_case)
    assert c.gold_subsumed is False and cert_case not in AG.CERTIFICATES
    assert AG.is_countermodel(c.tbox(), c.sub, c.sup, my_certificate)

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader and the Claude reviewer

The guidelines the scorer can report — ids plus the sentence GEPA's reflection reads —
are in `AG.DL_RULEBOOK`:

In [ ]:
for rule in AG.DL_RULEBOOK:
    print(f"{rule.id:38s} {rule.description[:80]}")

### B1. A diagnostic review scorer (14 pts)

Implement `dl_scorer(gold, pred) -> ev.ScoreReport`. `gold` is a dataset row
(`gold_dl`, `gold_subsumption`, `constructors`, `sub_satisfiable`, `sub`, `sup`); `pred`
has string fields `dl` and `subsumption`. The score is **0.5 for the name + 0.5 for the
verdict**.

**Name.** Normalise (strip, upper-case, remove spaces, drop a trailing `(D)`) and parse
against the standard order `^(ALC|S)(H?)(O?)(I?)([QNF]?)$`. Full marks only when the
parsed letters equal gold's. When wrong, report every rule that applies:

| situation | violated |
|---|---|
| does not parse (prose, wrong letter order, …) | `name-the-logic-exactly` |
| base differs (ALC vs S) | `s-for-transitive` |
| H / I needed but missing | `h-for-role-hierarchy` / `i-for-inverse` |
| gold has Q or N and the answer's number letter differs | `q-for-qualified-number` |
| a letter (H, O, I, or a number letter) that gold does not have | `no-unused-letters` |
| wrong, but none of the above applies | `name-the-logic-exactly` |

**Verdict.** Accept `true/yes/subsumed/entailed` and `false/no/not subsumed/not entailed`
(case-insensitive, surrounding punctuation ignored). Anything else earns 0 and
`answer-true-or-false`. A wrong verdict earns 0 and `use-the-reasoner`, plus
`unsatisfiable-subsumed-by-everything` when gold is *subsumed* because the subject class
is unsatisfiable.

The notes must say what was wrong **and why** — for a wrong name, the correct name and
the constructors the module uses. That text is what GEPA reads.

In [ ]:
DL_PATTERN = re.compile(r"^(ALC|S)(H?)(O?)(I?)([QNF]?)$")

def parse_dl(text) -> dict | None:
    # TODO: normalise, then match DL_PATTERN; return the letters, or None
    raise NotImplementedError

def dl_scorer(gold, pred) -> ev.ScoreReport:
    # TODO
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=11):
    by_id = {ex.id: ex for ex in train + dev + test}
    P = lambda d, s: dspy.Prediction(dl=d, subsumption=s)
    cases = [
        ("interlocking-control-centre", "SHIQ", "false", 1.0, []),
        ("interlocking-control-centre", " shiq(D) ", "False.", 1.0, []),
        ("interlocking-control-centre", "ALCHIQ", "no", 0.5, ["s-for-transitive"]),
        ("interlocking-control-centre", "SHIN", "false", 0.5, ["q-for-qualified-number"]),
        ("interlocking-control-centre", "SIQ", "false", 0.5, ["h-for-role-hierarchy"]),
        ("interlocking-control-centre", "SHOIQ", "false", 0.5, ["no-unused-letters"]),
        ("interlocking-control-centre", "The logic is SHIQ", "false", 0.5, ["name-the-logic-exactly"]),
        ("interlocking-control-centre", "SHIQ", "true", 0.5, ["use-the-reasoner"]),
        ("interlocking-control-centre", "SHQ", "probably", 0.0, ["i-for-inverse", "answer-true-or-false"]),
        ("bay-platform-low-use", "ALCQ", "false", 0.5, ["q-for-qualified-number"]),
        ("bay-platform-low-use", "ALC", "false", 0.5, ["q-for-qualified-number"]),
        ("barrier-safety-critical", "ALCN", "true", 0.5, ["no-unused-letters"]),
        ("barrier-safety-critical", "S", "yes", 0.5, ["s-for-transitive"]),
        ("automated-manual-crossing", "ALC", "false", 0.5,
         ["use-the-reasoner", "unsatisfiable-subsumed-by-everything"]),
        ("drainage-culvert", "ALC", "true", 0.5, ["use-the-reasoner"]),
    ]
    for item, d, s, score, violated in cases:
        r = dl_scorer(by_id[item], P(d, s))
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (item, d, s, r.score, r.violated)
        assert all(v in AG.DL_RULEBOOK for v in r.violated)
    wrong = dl_scorer(by_id["interlocking-control-centre"], P("ALCHIQ", "false"))
    assert any("SHIQ" in n and "transitive" in n for n in wrong.notes), "say the right name and why"

### B2. The reviewer as a DSPy program (7 pts)

Write a signature `ModuleReview` with inputs `kb` and `query` and outputs
`justification`, `dl` and `subsumption` (all strings), and a factory
`DLReviewer(instruction)` returning a `dspy.Module` with a single `dspy.Predict` whose
instruction is `instruction`. The field descriptions are part of the prompt — write them
for a colleague who has never seen this notation (manual marks), and say in a sentence why
you put the output fields in the order you did. Configure Claude and run it once on
`train[0]`.

In [ ]:
BASELINE_INSTRUCTION = ("You review ontology modules for a railway. Say which description "
                        "logic the module needs and answer the subsumption question.")

# TODO: class ModuleReview(dspy.Signature): ...
# TODO: def DLReviewer(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: DLReviewer()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=4):
    assert set(ModuleReview.input_fields) == {"kb", "query"}
    assert {"dl", "subsumption", "justification"} <= set(ModuleReview.output_fields)
    program = DLReviewer("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert isinstance(smoke.dl, str) and isinstance(smoke.subsumption, str)

**Your answer:**

*(Write your answer here.)*

### B3. Baseline on the development split, with its cost (8 pts)

Evaluate the baseline reviewer on `dev` with `ev.evaluate_dataset`, inside
`llm.meter(lm)`. Store the result in `baseline_dev` and the cost in `baseline_dev_cost`,
and build `b3`: the per-item rows joined with each item's `phenomenon` and `gold_dl`.

Then write a short **error analysis** separating the two halves of the task: which
*naming* rules were broken and on which phenomena, and which *verdicts* were wrong — and
for those, was it a definition, an inverse, or the unsatisfiable subject? Which of the two
halves could a tool fix completely, and which needs the model to reason?

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ..., b3 = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert baseline_dev["n"] == len(dev) == 8
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)
    assert isinstance(b3, pd.DataFrame) and len(b3) == 8 and {"phenomenon", "score"} <= set(b3.columns)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise honestly — then let the oracle label the training data

GEPA rewrites the instruction by reflecting on the scorer's feedback. The rules: **optimise
on `train`, select on `dev`, report on `test`**, and put a cost and a noise estimate next
to every number. Then use the chapter's distinctive asset — an oracle that labels for free
— to improve the reviewer from unlabelled traffic, behind a promotion gate.

### C1. A budgeted GEPA run with a held-out report (12 pts)

1. Build the GEPA feedback metric from your scorer and `AG.DL_RULEBOOK`, and a separate
   reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare baseline and tuned on **`test`** with `opt.compare` (store as `c1`) and print
   `c1.report()`.
4. Save the tuned instruction to `config.artifacts_dir() /
   "ch03_module_reviewer_instruction.txt"`.

In writing: read the instruction diff. Which guidelines did GEPA put into words? Did it
learn to *follow definitions* and to treat an unsatisfiable subject as subsumed — or only
the letter rules? Did it add anything that only fits the eight training modules?

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = config.artifacts_dir() / "ch03_module_reviewer_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=6):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 100

**Your answer:**

*(Write your answer here.)*

### C2. Is the difference bigger than the noise? (7 pts)

With caching **disabled** (`fresh = llm.dspy_lm(cache=False)`, `with dspy.context(lm=fresh):`),
run the baseline and the tuned reviewers on `test` three times each, inside
`llm.meter(fresh)`; store mean scores in `runs = {"baseline": [...], "gepa": [...]}`.
Set `verdict_c2` to `"significant"` if `|mean_gepa − mean_baseline| > 2·max(sd_gepa,
sd_baseline)`, else `"not significant"`.

In writing: what does this say about the C1 number, and what would a credible evaluation
of the reviewer need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs (inside llm.meter(fresh) -> c2_cost), then compute verdict_c2
verdict_c2 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=4):
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    _m = {k: st.mean(v) for k, v in runs.items()}
    _s = {k: st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c2 == _expected
    assert {"calls", "usd"} <= set(c2_cost)

**Your answer:**

*(Write your answer here.)*

### C3. Self-improvement with a free oracle — and a certificate (12 pts)

In production the requests arrive **unlabelled** (`AG.field_requests()` — no gold
fields). The tableau can label them, so the reviewer can learn from its own failures with
no human in the loop. Do it, but only with labels you can trust.

1. Implement `self_label(request, tbox=None) -> dspy.Example`. Use **only the engine**
   (never a stored label): the module is `tbox` or `AG.case(request.id).tbox()`. Return a
   new example with the request's fields plus `gold_dl`, `gold_subsumption`,
   `constructors`, `sub_satisfiable`, `labelled_by="tableau"` and `certified`: true when
   the verdict is *subsumed*, or the module is pure ALC (`AG.ALC_CONSTRUCTORS`), or a
   countermodel in `AG.CERTIFICATES` checks out. Keep `kb` and `query` as the inputs.
2. Label the field requests; keep the certified ones as `trusted`.
3. Wrap the reviewer as a `Skill` (name, description, `build=DLReviewer`,
   `scorer=dl_scorer`, `dataset=dev`, the baseline instruction), evaluate it, and build a
   `SelfImprovingSkill` with `holdout=dev`, a GEPA optimiser with `max_metric_calls=30`
   and `min_gain=0.05`. Inside `llm.meter(lm, reflect)` (store `c3_cost`): run it on
   `trusted`, run **one** improvement round, then a **control round** with the optimiser
   replaced by the identity (`sis.optimise = lambda program, trainset: program`).
4. Print `sis.report()` and `skill.card()`; save the skill to
   `config.artifacts_dir() / "ch03_rail_module_reviewer_skill.json"` as `skill_path`.

In writing: what exactly did the oracle remove from the usual self-improvement problem,
and what did it *not* remove? What did the control round show, and could it ever have
been promoted?

In [ ]:
def self_label(request, tbox: dl.TBox | None = None) -> dspy.Example:
    # TODO
    raise NotImplementedError

labelled_field = None   # TODO
trusted = None          # TODO
skill = None            # TODO
sis = None              # TODO: one GEPA round, then the identity control round
skill_path = config.artifacts_dir() / "ch03_rail_module_reviewer_skill.json"

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=6):
    for ex in train + dev + test:          # the labeller must reproduce every gold label from the module
        bare = dspy.Example(id=ex.id, kb=ex.kb, query=ex.query, sub=ex.sub, sup=ex.sup).with_inputs("kb", "query")
        lab = self_label(bare)
        assert (lab.gold_dl, lab.gold_subsumption, lab.sub_satisfiable) == \
            (ex.gold_dl, ex.gold_subsumption, ex.sub_satisfiable), ex.id
        assert lab.certified and lab.labelled_by == "tableau" and set(lab.inputs().keys()) == {"kb", "query"}
    _t, _sub, _sup = blind_spot
    _blind = dspy.Example(id="a2-blind-spot", kb=AG.describe_kb(_t), query="?", sub=_sub,
                          sup=_sup).with_inputs("kb", "query")
    assert self_label(_blind, tbox=_t).certified is False, "an uncertified 'no' must not be trusted"
    assert len(labelled_field) == len(AG.field_requests())
    assert all(not hasattr(r, "gold_dl") or r.gold_dl is None for r in AG.field_requests())
    assert len(sis.experience) == len(trusted)
    assert all(e.gold.labelled_by == "tableau" and e.gold.certified for e in sis.experience.episodes)
    assert {e.id for e in sis.holdout} == {e.id for e in dev}, "the gate reads dev, never test"
    assert len(sis.rounds) == 2
    for r in sis.rounds:
        assert r.promoted == ((r.after - r.before) >= sis.min_gain), f"round {r.round_index}"
    assert skill.version == 1 + sum(r.promoted for r in sis.rounds)
    assert skill_path.is_file() and {"calls", "usd"} <= set(c3_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part D — A review agent, and the price of soundness

The agent gets the reasoner as tools. `check_subsumption` is the *expensive, sound* call —
in production each one is a reasoner job on the integrated ontology, billed in CI minutes.

In [ ]:
ws_demo = AG.ReviewWorkspace("viaduct-critical")
demo_tools = {t.name: t for t in AG.build_review_tools(ws_demo)}
for name, t in demo_tools.items():
    print(f"{name:22s} {t.description.splitlines()[0]}")
print()
print(demo_tools["dl_expressivity"].invoke({}))
print(demo_tools["check_subsumption"].invoke({"sub": "Viaduct", "sup": "CriticalStructure"}))
print("trajectory:", ws_demo.log.names())

### D1. Build and evaluate the review agent (10 pts)

1. Write `REVIEW_PROMPT`, the agent's system prompt. It must make the agent inspect the
   module, name the logic from the constructors, check whether the subject class is
   satisfiable, **call `check_subsumption` before answering**, and end with a JSON object
   `{"dl": "...", "subsumed": true | false, "evidence": [...]}`.
2. Write `parse_review(text) -> dict` returning `{"dl": str, "subsumption": "true" |
   "false" | "unparseable"}` from the agent's final text (the JSON may be fenced or
   surrounded by prose; take the last object that has a `subsumed` key).
3. For every **test** item: a fresh `AG.ReviewWorkspace(ex.id)`, its tools, an agent
   (`agents.build_agent(ws, system_prompt=..., tools=...)`), one run on `ex.query`. Collect
   `d1` with columns `id`, `gold_dl`, `gold`, `predicted_dl`, `predicted`, `score` (your
   `dl_scorer` on the parsed answer), `reasoner_calls` (number of `check_subsumption`
   calls), `tool_calls` and `usd_estimate` (`llm.chat_usage(run.messages)`).

In writing: compare the agent with the tuned DSPy reviewer on the same test items — score
**and** cost per request. Which half of the task did the tools fix? Where did the agent
still go wrong, and was it the agent or the reasoner?

In [ ]:
REVIEW_PROMPT = """TODO"""

def parse_review(text: str) -> dict:
    # TODO
    raise NotImplementedError

d1 = None    # TODO: run the agent on every test item

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=5):
    assert parse_review('{"dl": "SHIQ", "subsumed": false, "evidence": []}') == {"dl": "SHIQ", "subsumption": "false"}
    assert parse_review('Done.\n```json\n{"dl": "ALC", "subsumed": true, "evidence": ["x"]}\n```')["subsumption"] == "true"
    assert parse_review('{"note": 1} and then {"dl": "S", "subsumed": "False", "evidence": []}') == {"dl": "S", "subsumption": "false"}
    assert parse_review("It is subsumed.")["subsumption"] == "unparseable"
    assert isinstance(d1, pd.DataFrame) and list(d1["id"]) == [ex.id for ex in test]
    assert {"id", "gold_dl", "gold", "predicted_dl", "predicted", "score", "reasoner_calls",
            "tool_calls", "usd_estimate"} <= set(d1.columns)
    assert set(d1["predicted"]) <= {"true", "false", "unparseable"}
    by_id = {ex.id: ex for ex in test}
    for _, row in d1.iterrows():
        expected = dl_scorer(by_id[row["id"]],
                             dspy.Prediction(dl=row["predicted_dl"], subsumption=row["predicted"])).score
        assert abs(row["score"] - expected) < 1e-9, row["id"]

**Your answer:**

*(Write your answer here.)*

### D2. Pricing the reasoner: a budgeted, stochastic MDP (10 pts)

The board caps reasoner jobs per batch. For each queued request the bot may **guess** with
the told-subsumption heuristic (free, right with probability `p`) or **reason** (always
right, reward `1 − c`, one unit of budget). `AG.ReasoningBudgetMDP` has exactly these
**stochastic** transitions.

1. **Estimate `p` from data.** When the heuristic says *subsumed* it is certain (A1), so
   `p = 1`. Otherwise `p` depends on the module's observable `AG.risk_class`: estimate
   `p_hat[class]` on **train + dev** as the fraction of heuristic-"no" items whose gold is
   *not subsumed*. Build `queue`, the `p` of each **test** item in order.
2. Solve `M = AG.ReasoningBudgetMDP(queue, reasoner_cost=COST, budget=BUDGET)` with
   `mdp.value_iteration`, and get `plan = M.optimal_plan(pi)`.
3. Derive the closed form and implement `rule_value(p, c, B)`: the optimal expected
   return. Verify it against value iteration on a sweep of costs and budgets, recorded in
   `d2` (columns `cost`, `budget`, `v_star`, `rule`, `reasoner_calls`).
4. The transitions are stochastic: check `V*` by Monte-Carlo (`mdp.policy_value`, 3000
   episodes) for the optimal policy and for *always guess*. Then compute `realised`: the
   return the optimal plan actually earns on the 8 test items, using the heuristic's real
   correctness.

In writing: state the decision rule; explain why "always call the reasoner" is not
optimal even with an unlimited budget; and compare `V*` with `realised`.

In [ ]:
COST, BUDGET = 0.25, 3

p_hat = None          # TODO (1): {risk_class: p}
queue = None          # TODO (1): one p per test item

def rule_value(p: list[float], c: float, B: int) -> float:
    # TODO (3)
    raise NotImplementedError

d2 = None             # TODO (3)
mc_optimal = mc_guess = realised = None   # TODO (4)

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=6):
    _pools = {}
    for ex in train + dev:
        t = AG.case(ex.id).tbox()
        if not told_subsumes(t, ex.sub, ex.sup):
            _pools.setdefault(AG.risk_class(t), []).append(not ex.gold_subsumption)
    assert set(p_hat) == set(_pools)
    assert all(abs(p_hat[k] - sum(v) / len(v)) < 1e-9 for k, v in _pools.items())
    assert len(queue) == len(test) and all(0.0 <= p <= 1.0 for p in queue)
    assert abs(rule_value([0.9, 0.5, 0.95, 0.6], 0.2, 2) - 3.45) < 1e-9
    assert abs(rule_value([0.9], 0.2, 1) - 0.9) < 1e-9 and abs(rule_value([0.3], 0.5, 0) - 0.3) < 1e-9
    assert abs(V[M.initial_state()] - rule_value(queue, COST, BUDGET)) < 1e-6
    assert isinstance(d2, pd.DataFrame) and len(d2) >= 20 and d2["budget"].nunique() >= 4
    assert ((d2.v_star - d2.rule).abs() < 1e-6).all(), "value iteration and the closed form disagree"
    for _, r in d2.iterrows():
        worth = sum(1 for p in queue if (1 - r.cost) > p)
        assert r.reasoner_calls == min(r.budget, worth), (r.cost, r.budget)
    assert abs(mc_optimal - V[M.initial_state()]) < 0.15 and abs(mc_guess - sum(queue)) < 0.15
    assert 0.0 <= realised <= len(test)

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 A told-subsumption heuristic, and a dataset that can punish it | 5 | 5 | 10 |
| A | A2 Where the free oracle stops being right | 4 | 6 | 10 |
| B | B1 A diagnostic review scorer | 11 | 3 | 14 |
| B | B2 The reviewer as a DSPy program | 4 | 3 | 7 |
| B | B3 Baseline on the development split, with its cost | 3 | 5 | 8 |
| C | C1 A budgeted GEPA run with a held-out report | 6 | 6 | 12 |
| C | C2 Is the difference bigger than the noise? | 4 | 3 | 7 |
| C | C3 Self-improvement with a free oracle — and a certificate | 6 | 6 | 12 |
| D | D1 Build and evaluate the review agent | 5 | 5 | 10 |
| D | D2 Pricing the reasoner: a budgeted, stochastic MDP | 6 | 4 | 10 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']
_ = GRADER.summary(AUTO_GRADED)